In [ ]:
import matplotlib.pyplot as plt
import struct
import pandas as pd

def parse_qbb_trace(file_path, max_rows=None):
    """
    Parses the binary QBB trace file (TraceFormat struct, 56 bytes each).

    This version matches the exact C++ struct definition and field layout.
    It decodes the union contents based on l3Prot:
      0x6   = TCP   -> data
      0x11  = UDP   -> data
      0xFC/0xFD = ACK
      0xFE  = PFC
      0xFF  = CNP
      else  = qp (default)
    """

    # --- Base struct before the union (12 fields) ---
    base_fmt = "<QHBBIIIHBBBB"
    base_size = struct.calcsize(base_fmt)

    # --- Union structs (from TraceFormat union) ---
    fmt_data = "<HHIQHH"      # sport, dport, seq, ts, pg, payload
    fmt_ack  = "<HHHHIQ"      # sport, dport, flags, pg, seq, ts
    fmt_pfc  = "<IIb3x"       # time, qlen, qIndex (+3 padding to 12 bytes)
    fmt_cnp  = "<HBBHH"       # fid, qIndex, ecnBits, qfb, total
    fmt_qp   = "<HH"          # sport, dport

    rec_size = 56  # confirmed from debugger
    rows = []

    # --- Read file into memory ---
    try:
        with open(file_path, "rb") as f:
            data = f.read()
    except FileNotFoundError:
        print(f"Error: file not found: {file_path}")
        return pd.DataFrame()

    n = len(data)
    if n < rec_size:
        print("Trace file too small — no complete records.")
        return pd.DataFrame()

    off = 0
    last_time = -1

    while off + rec_size <= n:
        # Stop if the maximum number of rows has been reached
        if max_rows is not None and len(rows) >= max_rows:
            print(f"Stopped after reading {max_rows} rows.")
            break
        
        # Unpack header (base)
        try:
            header = struct.unpack_from(base_fmt, data, off)
        except struct.error:
            break

        (time_ns, node, intf, qidx, qlen,
         sip, dip, size, l3Prot, event, ecn, nodeType) = header

        union_bytes = data[off + base_size : off + rec_size]

        row = {
            "time": time_ns / 1e9,
            "node": node,
            "intf": intf,
            "qidx": qidx,
            "qlen": qlen,
            "sip": sip,
            "dip": dip,
            "size": size,
            "l3Prot": l3Prot,
            "event": event,
            "ecn": ecn,
            "nodeType": nodeType,
            "sport": None,
            "dport": None,
            "seq": None,
            "ts": None,
            "pg": None,
            "payload": None,
            "ProtType": None,
        }

        # --- Decode union based on l3Prot ---
        try:
            if l3Prot in (0x6, 0x11):  # TCP/UDP data
                (sport, dport, seq, ts, pg, payload) = struct.unpack(fmt_data, union_bytes[:struct.calcsize(fmt_data)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "seq": seq,
                    "ts": ts,
                    "pg": pg,
                    "payload": payload,
                    "ProtType": "TCP" if l3Prot == 0x6 else "UDP",
                })
            elif l3Prot in (0xFC, 0xFD):  # ACK
                (sport, dport, flags, pg, seq, ts) = struct.unpack(fmt_ack, union_bytes[:struct.calcsize(fmt_ack)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "seq": seq,
                    "ts": ts,
                    "pg": pg,
                    "ProtType": "ACK",
                })
            elif l3Prot == 0xFE:  # PFC
                (pfc_time, pfc_qlen, qIndex) = struct.unpack(fmt_pfc, union_bytes[:struct.calcsize(fmt_pfc)])
                row.update({
                    "pfc_time": pfc_time,
                    "pfc_qlen": pfc_qlen,
                    "pfc_qIndex": qIndex,
                    "ProtType": "PFC",
                })
            elif l3Prot == 0xFF:  # CNP
                (fid, qIndex, ecnBits, qfb, total) = struct.unpack(fmt_cnp, union_bytes[:struct.calcsize(fmt_cnp)])
                row.update({
                    "cnp_fid": fid,
                    "cnp_qIndex": qIndex,
                    "cnp_ecnBits": ecnBits,
                    "cnp_qfb": qfb,
                    "cnp_total": total,
                    "ProtType": "CNP",
                })
            else:  # default qp
                (sport, dport) = struct.unpack(fmt_qp, union_bytes[:struct.calcsize(fmt_qp)])
                row.update({
                    "sport": sport,
                    "dport": dport,
                    "ProtType": "QP",
                })
        except struct.error:
            # corrupted or incomplete record
            pass

        rows.append(row)

        if time_ns < last_time:
            # likely misaligned data — stop
            break
        last_time = time_ns
        off += rec_size

    return pd.DataFrame(rows)

def calculate_throughput(df, interval):
    """
    Calculates throughput in packets over a specified time interval.
    """
    if df.empty:
        return pd.Series()
        
    # Set time as index
    df = df.set_index(pd.to_datetime(df['time'], unit='s'))
    
    nanoseconds_interval = int(interval * 1e9)
    freq_str = f'{nanoseconds_interval}N'

    # Resample data into time bins and sum the packet count
    throughput = df['size'].resample(freq_str).count()
    if not df.empty:
        total_bytes_per_interval = df['size'].resample(freq_str).sum()
        throughput_gbps = (total_bytes_per_interval * 8) / (interval * 1e9)
    else:
        throughput_gbps = throughput * 0 # Return a series of zeros with the correct index

    return throughput_gbps

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot(df, interval):
    # === Configuration ===
    group_by_node = True      # Toggle: True = per-node plots, False = per-flow plots
    
    # === Build mapping from sip -> node (node is the shorter sender id) ===
    sip_node_map = (
        df[['sip', 'node']]
        .drop_duplicates(subset='sip')
        .set_index('sip')['node']
        .to_dict()
    )

    # === Create short-name columns ===
    df['sip_short'] = df['sip'].map(sip_node_map).fillna(df['sip']).astype(str)
    df['dip_short'] = df['dip'].map(sip_node_map).fillna(df['dip']).astype(str)

    # === Event labels ===
    event_labels = {0: "Recv", 1: "Enqu", 2: "Dequ"}

    # === Helper: group column selection ===
    base_group_cols = ['sip_short', 'dip_short', 'ProtType', 'sport', 'dport']
    if group_by_node:
        group_cols = ['node'] + base_group_cols
        print("🔹 Grouping by node (per-node throughput plots).")
    else:
        group_cols = base_group_cols
        print("🔹 Grouping by flow only (aggregated throughput).")

    # === Loop over events ===
    for event_val, event_name in event_labels.items():
        df_event = df[df['event'] == event_val]
        if df_event.empty:
            print(f"No data for event {event_name} ({event_val}).")
            continue

        series_list = []
        count_dict = {}

        # Group by selected columns
        for keys, group_df in df_event.groupby(group_cols):
            if group_by_node:
                node, sip_s, dip_s, prot, sport, dport = keys
                key_name = f"{node}_{sip_s}_to_{dip_s}_prot{str(prot).replace('/', '_')}_sport{sport}_dport{dport}"
            else:
                sip_s, dip_s, prot = keys
                key_name = f"{sip_s}_to_{dip_s}_prot{str(prot).replace('/', '_')}"

            # Count raw entries
            count_dict[key_name] = len(group_df)

            # Compute throughput series
            s = calculate_throughput(group_df, interval)
            if s.empty:
                continue
            series_list.append(s.rename(key_name))

        # Print raw entry counts
        print(f"\n=== Entry counts for event '{event_name}' (raw dataset rows) ===")
        for k, v in count_dict.items():
            print(f"{k}: {v}")

        if not series_list:
            print(f"No throughput series to plot for event {event_name}.")
            continue

        df_all = pd.concat(series_list, axis=1).fillna(0)

        # === Plotly Figure ===
        fig = go.Figure()

        for col in df_all.columns:
            count = count_dict.get(col, 0)
            fig.add_trace(go.Scatter(
                x=df_all.index,
                y=df_all[col],
                mode='lines',
                name=f"{col} ({count} entries)"
            ))

        grouping_text = "Per-Node" if group_by_node else "Per-Flow"
        fig.update_layout(
            title=f"{grouping_text} Throughput Over Time ({event_name})",
            xaxis_title="Time",
            yaxis_title="Throughput",
            hovermode='x unified',
            legend_title="Flow (and Node, if applicable)",
            template='plotly_white',
            height=600,
            width=1000
        )

        # === Save and show ===
        base_name = globals().get('base', 'throughput')
        out_html = f"{base_name}_{event_name}.html"
        fig.write_html(out_html)
        print(f"\n✅ Interactive plot for event '{event_name}' saved to {out_html}")
        fig.show()


In [ ]:
# trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy/run_20251113_235331/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251116_221612/ns3/astrasim_trace.tr'
trace_output_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/multiple_collectives/all_gather_size_33554432_group_0/run_20251208_212044_738ms/ns3/astrasim_trace.tr'
df = parse_qbb_trace(trace_output_file, 3000000)
# interval = 0.1


In [ ]:
sip_node_map = (
    df[['sip', 'node']]
    .drop_duplicates(subset='sip')
    .set_index('sip')['node']
    .to_dict()
)
# === Create short-name columns ===
df['sip_short'] = df['sip'].map(sip_node_map).fillna(df['sip']).astype(str)
df['dip_short'] = df['dip'].map(sip_node_map).fillna(df['dip']).astype(str)


In [ ]:
a = df[(df['sip_short'] == '7')& (df['dip_short'] == '5')& (df['ProtType'] == 'ACK')& (df['event'] == 2)&(df['node']==21)]
event_labels = {0: "Recv", 1: "Enqu", 2: "Dequ"}

In [ ]:
a.head(20)

In [ ]:
b = df[(df['sip_short'] == '7')& (df['dip_short'] == '5')& (df['ProtType'] == 'ACK')& (df['event'] == 2)&(df['node']==7)]
ba = b.iloc[[i for i in range(0,len(b),3)],:]

ba['dt'] = ba['time'].shift(-1) - ba['time']
ba.head(100)

In [ ]:
c = df[((df['sip_short'] == '4')| (df['sip_short'] == '5'))& (df['ProtType'] == 'UDP')& (df['event'] == 2)&(df['node']==20)]

In [ ]:
pd.set_option('display.max_rows', 500)

In [ ]:
c.head(28)

In [ ]:
plot(df, 0.000005)

In [ ]:
import pandas as pd
from typing import List, Optional, Dict

def extract_routes_from_trace(df: pd.DataFrame) -> Dict[str, List[int]]:
    """
    Extracts packet routes from a trace DataFrame.
    It focuses on tracing the path of packets from source to destination
    by observing Enqueue and Receive events.

    Args:
        df: The DataFrame from parse_qbb_trace.

    Returns:
        A dictionary mapping 'src:dst' to a list of node IDs in the path.
    """
    if df.empty:
        return {}

    # We need Enqueue (1) and Receive (0) events to trace a full path.
    df_path = df[df['event'].isin([0])&(df['l3Prot'].isin([17]))].copy()

    # The source node for a flow is where the packet is first enqueued.
    sip_node_map = (
        df[['sip', 'node']]
        .drop_duplicates(subset='sip')
        .set_index('sip')['node']
        .to_dict()
    )

    routes = {}
    # Group by each flow (source IP, destination IP) to trace its path.
    for (sip, dip), flow_df in df_path.groupby(['sip', 'dip']):
        src_node = sip_node_map.get(sip)
        dst_node = sip_node_map.get(dip)

        if src_node is None:
            continue

        # Filter for Enqueue events to trace the path through switches,
        # and sort by time to get the correct order.
        path_events = flow_df[flow_df['event'] == 0].sort_values('time')
        
        path = [src_node]
        last_node = -1
        # Iterate through the Enqueue events and record the node ID for each hop.
        # We only add a node if it's different from the previous one.
        for node in path_events['node'].unique():
            if node != last_node:
                path.append(int(node))
                last_node = node
        
        # The final hop is the destination node where the packet is received.
        # Add it to the path if it's not already the last node.
        if dst_node != last_node:
            path.append(dst_node)
        
        # A valid path must start at the source and end at the destination.
        if path and path[0] == src_node and path[-1] == dst_node:
            route_key = f"{src_node}:{dst_node}"
            # We only store the first valid path found for a given src:dst pair.
            if route_key not in routes:
                routes[route_key] = path
            
    return routes

In [ ]:
import os

def generate_topology_with_new_routes(base_topology_path, output_topology_path, routes_dict):
    """
    Creates a new topology file by replacing the ROUTES section of a base file.

    Args:
        base_topology_path (str): Path to the source topology file.
        output_topology_path (str): Path where the new topology file will be saved.
        routes_dict (dict): A dictionary of routes to write into the new file.
    """
    try:
        with open(base_topology_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: Base topology file not found at {base_topology_path}")
        return

    # Find the start of the ROUTES section
    try:
        routes_start_index = [i for i, line in enumerate(lines) if 'ROUTES' in line][0]
        # Keep everything before the ROUTES section
        new_content = lines[:routes_start_index]
    except IndexError:
        print(f"Warning: 'ROUTES' section not found in {base_topology_path}. Appending to the end.")
        new_content = lines
        
    # Add the new ROUTES header
    new_content.append("ROUTES\n")

    # Sort and add the new routes from the dictionary
    sorted_keys = sorted(
        routes_dict.keys(), 
        key=lambda x: tuple(map(int, x.split(':')))
    )
    for key in sorted_keys:
        path_str = ', '.join(map(str, routes_dict[key]))
        new_content.append(f"{key}:[{path_str}]\n")

    # Write the new topology file
    try:
        with open(output_topology_path, 'w') as f:
            f.writelines(new_content)
        print(f"✅ Successfully created new topology file: {output_topology_path}")
    except IOError as e:
        print(f"Error writing to {output_topology_path}: {e}")


# --- Main execution ---

# 1. Define the run folders and the base topology file
run_folders = [
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251117_101037',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251117_101128',
    '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251117_101216'
]
run_folder = ['/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251117_135940']
base_topology_file = '/app/astra-sim/upc/configuration/ns3/FoldedClos_16_topology.txt'
output_dir = '/app/astra-sim/upc/configuration/ns3/'

# 2. Loop through each folder, extract routes, and generate a new topology file
for folder in run_folders:
    run_name = os.path.basename(folder)
    trace_file = os.path.join(folder, 'ns3', 'astrasim_trace.tr')
    
    print(f"\n--- Processing run: {run_name} ---")

    if not os.path.exists(trace_file):
        print(f"Warning: Trace file not found at {trace_file}")
        continue

    # Parse trace and extract routes
    df_run = parse_qbb_trace(trace_file)
    if df_run.empty:
        print(f"Warning: No data parsed from {trace_file}. Skipping.")
        continue
        
    extracted_routes = extract_routes_from_trace(df_run)
    if not extracted_routes:
        print(f"Warning: No routes could be extracted from {trace_file}. Skipping.")
        continue

    # --- Generate .txt topology file ---
    new_topology_filename_txt = f"FoldedClos_16_topology_{run_name}.txt"
    output_path_txt = os.path.join(output_dir, new_topology_filename_txt)
    generate_topology_with_new_routes(base_topology_file, output_path_txt, extracted_routes)

    # --- Generate .json topology file ---
    base_json_topology_file = '/app/astra-sim/upc/configuration/g2/FoldedClos_16_topology.json'
    output_json_dir = '/app/astra-sim/upc/configuration/g2/'
    new_topology_filename_json = f"FoldedClos_16_topology_g2_{run_name}.json"
    output_path_json = os.path.join(output_json_dir, new_topology_filename_json)
    
    # Create the output directory for JSON if it doesn't exist
    os.makedirs(output_json_dir, exist_ok=True)
    
    generate_json_topology_with_new_routes(base_json_topology_file, output_path_json, extracted_routes)


In [ ]:
import json

def generate_json_topology_with_new_routes(base_topology_path, output_topology_path, routes_dict):
    """
    Creates a new JSON topology file by replacing the paths with new routes,
    applying a specific integer ID to string name mapping.

    Args:
        base_topology_path (str): Path to the source JSON topology file.
        output_topology_path (str): Path where the new JSON topology file will be saved.
        routes_dict (dict): A dictionary of routes with integer node IDs.
    """
    try:
        with open(base_topology_path, 'r') as f:
            topology_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: Base JSON topology file not found at {base_topology_path}")
        return
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from {base_topology_path}: {e}")
        return

    def map_id_to_name(node_id):
        """Maps an integer node ID to its string name (e.g., 0 -> 'h1', 16 -> 's1')."""
        if 0 <= node_id <= 15:
            return f"h{node_id + 1}"
        elif 16 <= node_id <= 35:
            return f"s{node_id - 15}"
        return str(node_id)

    # Clear existing paths to ensure only the new ones are added
    topology_data['paths'] = {}

    for route_key, path_ids in routes_dict.items():
        try:
            src_id_str, dst_id_str = route_key.split(':')
            src_id, dst_id = int(src_id_str), int(dst_id_str)
        except ValueError:
            print(f"Warning: Could not parse route key '{route_key}'. Skipping.")
            continue

        src_name = map_id_to_name(src_id)
        dst_name = map_id_to_name(dst_id)
        
        # Convert the list of integer path IDs to their string names
        path_names = [map_id_to_name(node_id) for node_id in path_ids]

        if src_name not in topology_data['paths']:
            topology_data['paths'][src_name] = {}
        
        # Add the new route, wrapped in a list as per the original JSON format
        topology_data['paths'][src_name][dst_name] = [path_names]

    # Write the updated topology data to the new JSON file
    try:
        with open(output_topology_path, 'w') as f:
            json.dump(topology_data, f, indent=4)
        print(f"✅ Successfully created new JSON topology file: {output_topology_path}")
    except IOError as e:
        print(f"Error writing to {output_topology_path}: {e}")

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import re

def parse_qlen_log(file_path):
    """
    Parses the queue length log file into a pandas DataFrame.
    
    The expected format for each line is:
    'time <timestamp> <node_id> j <iface1> <qlen1> j <iface2> <qlen2> ...'
    """
    records = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts or parts[0] != 'time':
                    continue
                
                time_ns = int(parts[1])
                node_id = int(parts[2])
                
                # Iterate over the 'j <iface> <qlen>' groups
                for i in range(3, len(parts), 3):
                    if parts[i] == 'j' and i + 2 < len(parts):
                        iface_id = int(parts[i+1])
                        qlen = int(parts[i+2])
                        records.append({
                            'time_ns': time_ns,
                            'time_s': time_ns / 1e9,
                            'node': node_id,
                            'interface': iface_id,
                            'qlen_bytes': qlen
                        })
    except FileNotFoundError:
        print(f"Error: Queue length file not found at {file_path}")
        return pd.DataFrame()
    except (ValueError, IndexError) as e:
        print(f"Error parsing file {file_path}: {e}")
        return pd.DataFrame()

    if not records:
        print("No valid queue length records found.")
        return pd.DataFrame()
        
    return pd.DataFrame(records)

def plot_qlen(df_qlen):
    """
    Generates an interactive plot of queue length over time for each switch interface.
    """
    if df_qlen.empty:
        print("DataFrame is empty, cannot plot queue lengths.")
        return

    fig = go.Figure()

    # Group data by switch node and interface to plot each as a separate line
    for (node, interface), group in df_qlen.groupby(['node', 'interface']):
        fig.add_trace(go.Scatter(
            x=group['time_s'],
            y=group['qlen_bytes'],
            mode='lines',
            name=f'Switch {node}, Interface {interface}'
        ))

    fig.update_layout(
        title='Switch Queue Length Over Time',
        xaxis_title='Time (seconds)',
        yaxis_title='Queue Length (Bytes)',
        hovermode='x unified',
        legend_title="Switch Interface",
        template='plotly_white',
        height=600,
        width=1000
    )

    # Save and show the plot
    out_html = "queue_length_plot.html"
    fig.write_html(out_html)
    print(f"\n✅ Interactive queue length plot saved to {out_html}")
    fig.show()

# --- Example Usage ---
# 1. Define the path to your queue length log file
qlen_file_path = '/app/astra-sim/upc/configuration/ns3/output/astrasim_16nodes_ring_pfc_qlen.txt'

# 2. Parse the file into a DataFrame
df_qlen = parse_qlen_log(qlen_file_path)

# 3. Plot the data
if not df_qlen.empty:
    plot_qlen(df_qlen)
else:
    print("Could not generate queue length plot.")

In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    Returns the full path to the first .txt file found, or None.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a simple 'key value' or 'key = value' configuration file into a dictionary.
    Ignores lines starting with '#' and handles potential whitespace.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                # Try splitting by '=' first, then by whitespace
                if '=' in line:
                    parts = line.split('=', 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params


def plot_recv_per_folder(
    run_folders: List[str], 
    interval: float, 
    group_by_node: bool = False, 
    nodes_to_plot: Optional[List[int]] = None,
    src_nodes_to_plot: Optional[List[int]] = None,
    dst_nodes_to_plot: Optional[List[int]] = None

):
    """
    Generates a separate plot of 'Recv' throughput for each run folder,
    with lines broken down by node and flow, and allows filtering by node.
    """
    for folder in run_folders:
        print(f"Processing folder: {folder}")

        # 1. Parse config to create a descriptive title
        config_file = find_config_file(folder)
        if not config_file:
            print(f"  - Warning: Config file not found in {folder}")
            continue
            
        config_params = parse_config(config_file)
        
        title_parts = [
            f"cc:{config_params.get('cc_mode', 'N/A')}",
            f"win:{config_params.get('has_win', 'N/A')}",
            f"adapt:{config_params.get('var_win', 'N/A')}",
            f"buf:{config_params.get('buffer_size', 'N/A')}",
            f"size:{config_params.get('packet_payload_size', 'N/A')}"
        ]
        plot_title = ', '.join(title_parts)

        # 2. Parse trace file
        trace_file = os.path.join(folder, 'ns3', 'astrasim_trace.tr')
        if not os.path.exists(trace_file):
            print(f"  - Warning: Trace file not found: {trace_file}")
            continue
            
        df = parse_qbb_trace(trace_file)
        if df.empty:
            print(f"  - Warning: No data parsed from {trace_file}")
            continue
            
        df_recv = df[df['event'] == 0].copy()
        
        if df_recv.empty:
            print(f"  - Warning: No 'Recv' events found in {trace_file}")
            continue

        # Filter by selected nodes if a list is provided
        if nodes_to_plot:
            df_recv = df_recv[df_recv['node'].isin(nodes_to_plot)]
            if df_recv.empty:
                print(f"  - Warning: No 'Recv' events found for specified nodes {nodes_to_plot} in {trace_file}")
                continue

        # 3. Group data by flow and calculate throughput for each
        sip_node_map = (
            df[['sip', 'node']]
            .drop_duplicates(subset='sip')
            .set_index('sip')['node']
            .to_dict()
        )

        df_recv['sip_short'] = df_recv['sip'].map(sip_node_map).fillna(df_recv['sip'])
        df_recv['dip_short'] = df_recv['dip'].map(sip_node_map).fillna(df_recv['dip'])

        # Filter by source, destination, and current nodes if lists are provided
        if src_nodes_to_plot:
            df_recv = df_recv[df_recv['sip_short'].isin(src_nodes_to_plot)]
        if dst_nodes_to_plot:
            # For 'Recv' events, the 'node' column is the destination
            df_recv = df_recv[df_recv['dip_short'].isin(dst_nodes_to_plot)]

        base_group_cols = ['sip_short', 'dip_short', 'ProtType']
        group_cols = ['node'] + base_group_cols if group_by_node else base_group_cols

        series_list = []
        for keys, group_df in df_recv.groupby(group_cols):
            if group_by_node:
                node, sip_s, dip_s, prot = keys
                key_name = f"Node {node}: {sip_s}→{dip_s} ({prot or 'N/A'}) {len(group_df)} pkts"
            else:
                sip_s, dip_s, prot = keys
                key_name = f"{sip_s}→{dip_s} ({prot or 'N/A'})"

            s = calculate_throughput(group_df, interval)
            if not s.empty:
                series_list.append(s.rename(key_name))

        if not series_list:
            print(f"  - Warning: No throughput data calculated for {trace_file}")
            continue
        
        df_all = pd.concat(series_list, axis=1).fillna(0)

        # 4. Create and show the plot for the current folder
        fig = go.Figure()
        for col in df_all.columns:
            fig.add_trace(go.Scatter(
                x=df_all.index,
                y=df_all[col],
                mode='lines',
                name=col
            ))
        
        grouping_text = "Per-Node" if group_by_node else "Per-Flow"
        fig.update_layout(
            title=f"Recv Throughput ({grouping_text})<br><sup>{plot_title}</sup>",
            xaxis_title="Time (seconds)",
            yaxis_title="Throughput (Gbps)",
            hovermode='x unified',
            legend_title="Flow",
            template='plotly_white',
            height=600,
            width=1000
        )
        
        folder_name_sanitized = os.path.basename(folder).replace('/', '_')
        # out_html = f"recv_throughput_per_flow_{folder_name_sanitized}.html"
        # fig.write_html(out_html)
        # print(f"  ✅ Interactive plot saved to {out_html}")
        fig.show()

# --- Usage ---
# base_run_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/sends_recv_easy'
# folders_to_plot = [os.path.join(base_run_folder, d) for d in os.listdir(base_run_folder) if os.path.isdir(os.path.join(base_run_folder, d))]
folders_to_plot = ['/app/astra-sim/upc/output/comparison_run/FoldedClos/multiple_collectives/all_gather_size_33554432_group_0/run_20251208_193902_047ms']
plot_interval = 0.000005
nodes_to_include = list(range(37))
source_nodes = [0,1,2,3,4,5,6,7] 
dest_nodes =[0,1,2,3,4,5,6,7]
plot_recv_per_folder(
    folders_to_plot, 
    plot_interval, 
    group_by_node=True, 
    nodes_to_plot=nodes_to_include,
    src_nodes_to_plot=source_nodes,
    dst_nodes_to_plot=dest_nodes
)


In [ ]:
folders_to_plot